In [1]:
import pysam
import numpy as np
import pandas as pd

In [2]:
def get_overlap(a: tuple, b: tuple) -> int:
    """ Calculates the overlap between two intervals.

    Args:
        a (tuple): Interval a = (start, end)
        b (tuple): Interval b = (start, end)

    Returns:
        int: Overlap between a and b
    """        
    
    return max(0, min(a[1], b[1]) - max(a[0], b[0]) + 1)

In [3]:
def get_clipped_span(read: pysam.AlignedSegment) -> tuple:
    """ Calculates the span of a read that is clipped.

    Args:
        read (pysam.AlignedSegment): Sequencing Read

    Returns:
        tuple: Reference start and end of the clipped part of the read
    """        
    
    ref_start = read.reference_start
    ref_end = read.reference_end
    cigar = read.cigartuples
    
    # Beginnging of read is clipped
    if cigar[0][0] == 4 or cigar[0][0] == 5:
        return (ref_start - cigar[0][1], ref_start - 1)
    
    # End of read is clipped
    if cigar[-1][0] == 4 or cigar[-1][0] == 5:
        return (ref_end + 1, ref_end + cigar[-1][1])
    
    return None

In [4]:
def calculate_read_based_features(bam: pysam.AlignmentFile, chrom: str, start: int, stop: int, suffix: str) -> pd.Series:
    """ Calculates read-based features for a region.

    Args:
        bam (pysam.AlignmentFile): BAM file
        chrom (str): Chromosome name
        start (int): Start position
        stop (int): End position
        suffix (str): Suffix for column names

    Returns:
        pd.Series: Series containing the read-based features
    """        
    
    # Manually set chromosomal average, this is usually calculated
    baseline_insertsize_median = 400
    baseline_insertsize_mad = 50
    baseline_mapq_mean = 60
    baseline_mapq_std = 10
    
    insert_sizes = []
    mapqs = []
    all_reads = 0
    all_reads_extended = 0
    clipped_reads = 0
    split_reads = 0
    disco_ff_reads = 0
    disco_rr_reads = 0
    disco_rf_reads = 0

    for read in bam.fetch(chrom, start-5, stop+5):
        # add 5 bp padding because clipped bases cannot be used to fetch reads from a BAM file
        if not read.is_unmapped:
            if not read.reference_end <= start and not read.reference_start >= stop:
                
                # only consider reads that overlap with the region for which we want to calculate the features
                insert_sizes.append(abs(read.template_length))
                mapqs.append(read.mapping_quality)
                all_reads += 1
                if read.has_tag('SA'):
                    split_reads += 1
                    
                # Read orientation for inversions
                if read.is_reverse and read.mate_is_reverse:
                    disco_rr_reads += 1
                if not read.is_reverse and not read.mate_is_reverse:
                    disco_ff_reads += 1
                
                # Read orientation for duplications
                if read.is_read1 and read.is_reverse and not read.mate_is_reverse:
                    disco_rf_reads += 1
                elif read.is_read2 and not read.is_reverse and read.mate_is_reverse:
                    disco_rf_reads += 1
                elif read.is_read2 and read.is_reverse and not read.mate_is_reverse:
                    disco_rf_reads += 1
                elif read.is_read1 and not read.is_reverse and read.mate_is_reverse:
                    disco_rf_reads += 1

            # for clipped reads, we needed to extend the region by 5 bp
            all_reads_extended += 1
            clip_span = get_clipped_span(read)
            if clip_span != None:
                overlap = get_overlap(clip_span, [start, stop])
                if overlap > 0:
                    clipped_reads += 1
        
    if all_reads > 0:
            # add 0.1 to avoid -inf values
        insertsize_mean = np.round(np.log2((np.mean(insert_sizes) + 0.1) / (baseline_insertsize_median + 0.1)), 3)
        insertsize_std = np.round(np.log2((np.std(insert_sizes) + 0.1) / (baseline_insertsize_mad + 0.1)), 3)

        mapping_quality_mean = np.round(np.log2((np.mean(mapqs) + 0.1) / (baseline_mapq_mean + 0.1)), 3)
        mapping_quality_std = np.round(np.log2((np.std(mapqs) + 0.1) / (baseline_mapq_std + 0.1)), 3)

        splitreads_proportion = np.round(split_reads / all_reads, 3)
        clippedreads_proportion = np.round(clipped_reads / all_reads_extended, 3)
        disco_ff_proportion = np.round(disco_ff_reads / all_reads, 3)
        disco_rr_proportion = np.round(disco_rr_reads / all_reads, 3)
        disco_rf_proportion = np.round(disco_rf_reads / all_reads, 3)
        
    else:
        # special case: no reads in region
        insertsize_mean = np.round(np.log2(0.1 / (baseline_insertsize_median + 0.1)), 3)
        insertsize_std = np.round(np.log2(0.1 / (baseline_insertsize_mad + 0.1)), 3)

        mapping_quality_mean = np.round(np.log2(0.1 / (baseline_mapq_mean + 0.1)), 3)
        mapping_quality_std = np.round(np.log2(0.1 / (baseline_mapq_std + 0.1)), 3)

        splitreads_proportion = 0
        clippedreads_proportion = 0
        disco_ff_proportion = 0
        disco_rr_proportion = 0
        disco_rf_proportion = 0

    return pd.Series([insertsize_mean, insertsize_std, mapping_quality_mean, mapping_quality_std, splitreads_proportion, clippedreads_proportion, disco_ff_proportion, disco_rr_proportion, disco_rf_proportion], index =['ill_isize_mean_' + suffix, 'ill_isize_std_' + suffix, 'ill_mapq_mean_' + suffix, 'ill_mapq_std_' + suffix, 'ill_splitreads_' + suffix, 'ill_clipreads_' + suffix, 'ill_disco_ff_' + suffix, 'ill_disco_rr_' + suffix, 'ill_disco_rf_' + suffix])

In [8]:
# Set parameters
bam_filename = '/confidential/tGenVar/tech/illumina/snakemake_results/bam_hg38/17_08/bwa_mem.pe.sorted.mdup.bam'
bam = pysam.AlignmentFile(bam_filename, 'rb')

df_calls = pd.read_csv('17-08_hg38_DEL_DUP.csv')
df_calls_annot = df_calls.copy()
df_calls_annot = df_calls_annot.head(10)

In [12]:
# Annotate read-based features, usually this is parallelized per sample, sv_type and chromosome
# New feature columns needed:
# - ill_splitreads_I_II
# - ill_splitreads_I_III
# - ill_splitreads_I_IV
# - ill_splitreads_II_III
# - ill_splitreads_II_IV
# - ill_splitreads_III_IV

df_calls_annot.loc[:, ['ill_isize_mean_I', 'ill_isize_std_I', 'ill_mapq_mean_I', 'ill_mapq_std_I', 
                       'ill_splitreads_I', 'ill_clipreads_I', 'ill_disco_ff_I', 'ill_disco_rr_I', 
                       'ill_disco_rf_I']] = df_calls_annot.apply(lambda x: calculate_read_based_features(bam, x['chrom'], x['start'] - 50, x['start'], 'I'), axis=1, result_type ='expand')

df_calls_annot.loc[:, ['ill_isize_mean_II', 'ill_isize_std_II', 'ill_mapq_mean_II', 'ill_mapq_std_II', 
                       'ill_splitreads_II', 'ill_clipreads_II', 'ill_disco_ff_II', 'ill_disco_rr_II', 
                       'ill_disco_rf_II']] = df_calls_annot.apply(lambda x: calculate_read_based_features(bam, x['chrom'], x['start'], x['start'] + 50, 'II'), axis=1, result_type ='expand')
        
df_calls_annot.loc[:, ['ill_isize_mean_III', 'ill_isize_std_III', 'ill_mapq_mean_III', 'ill_mapq_std_III', 
                       'ill_splitreads_III', 'ill_clipreads_III', 'ill_disco_ff_III', 'ill_disco_rr_III', 
                       'ill_disco_rf_III']] = df_calls_annot.apply(lambda x: calculate_read_based_features(bam, x['chrom'], x['end'] - 50, x['end'], 'III'), axis=1, result_type ='expand')

df_calls_annot.loc[:, ['ill_isize_mean_IV', 'ill_isize_std_IV', 'ill_mapq_mean_IV', 'ill_mapq_std_IV', 
                       'ill_splitreads_IV', 'ill_clipreads_IV', 'ill_disco_ff_IV', 'ill_disco_rr_IV', 
                       'ill_disco_rf_IV']] = df_calls_annot.apply(lambda x: calculate_read_based_features(bam, x['chrom'], x['end'], x['end'] + 50, 'IV'), axis=1, result_type ='expand')